In [ ]:
import numpy as np
import pandas as pd
import tab01
from IPython.display import clear_output
from joblib import Parallel, delayed

from drift_diffusion.sim import sample_from_pdf

In [ ]:
# true parameters and sample sizes to test
params = {"a": 0.63, "t0": 0.435, "v": 2.23, "z": 0.008}
sample_sizes, n_repeats = [100, 500, 1000], 50

In [ ]:
# !! x min runtime if refit !!
REFIT = True  # True to refit model comparison, False to load precomputed results


@delayed
def run_simulation(rep, n):
    X = pd.DataFrame({"intercept": np.ones(n)})
    y = sample_from_pdf(**params, n_samples=n, random_state=rep + n)
    y_df = pd.DataFrame({"rt": np.abs(y), "response": np.sign(y)})

    mle_runtime, mle_params_, mle_unc_ = tab01.fit_mle(X, y)
    mcmc_runtime, mcmc_params_, mcmc_unc_ = tab01.fit_mcmc(y_df)

    return {
        "mle runtime (s)": mle_runtime,
        "mcmc runtime (s)": mcmc_runtime,
        "mle params": mle_params_,
        "mle unc": mle_unc_,
        "mcmc params": mcmc_params_,
        "mcmc unc": mcmc_unc_,
    }


# compare both models across sample sizes using normalized absolute error
if REFIT:
    with Parallel(n_jobs=-4) as parallel:
        df = []
        true_params = np.array(list(params.values()))
        for n in sample_sizes:
            results = parallel(run_simulation(rep, n) for rep in range(n_repeats))
            clear_output()
            mle_params_ = np.array([res["mle params"] for res in results])
            mcmc_params_ = np.array([res["mcmc params"] for res in results])
            mle_unc_ = np.array([res["mle unc"] for res in results])
            mcmc_unc_ = np.array([res["mcmc unc"] for res in results])
            mle_true_unc = mle_params_.std(axis=0)
            mcmc_true_unc = mcmc_params_.std(axis=0)
            mle_runtime = np.mean([res["mle runtime (s)"] for res in results])
            mcmc_runtime = np.mean([res["mcmc runtime (s)"] for res in results])
            df.append(
                {
                    "n_trials": n,
                    "mle runtime (s)": mle_runtime,
                    "mcmc runtime (s)": mcmc_runtime,
                    "mcmc/mle runtime": mcmc_runtime / mle_runtime,
                    "mle param": tab01.get_parameter_bias(mle_params_, true_params),
                    "mcmc param": tab01.get_parameter_bias(mcmc_params_, true_params),
                    "mle unc": tab01.get_uncertainty_bias(mle_unc_, mle_true_unc),
                    "mcmc unc": tab01.get_uncertainty_bias(mcmc_unc_, mcmc_true_unc),
                }
            )
        df_tab01 = pd.DataFrame(df).round(3)

else:
    df_tab01 = pd.read_csv("./results/tab-mle-vs-mcmc.csv")

In [ ]:
df_tab01["mcmc/mle runtime"] = df_tab01["mcmc runtime (s)"] / df_tab01["mle runtime (s)"]
df_tab01